# LangGraph Code Generation Agent

This notebook tests and demonstrates the LangGraph-based code generation agent from `agent.py`.

The agent:
- Generates Python code based on natural language queries
- Validates and executes the generated code
- Retries up to 3 times if execution fails
- Supports both `exec` and `subprocess` execution methods

In [1]:
# Import the agent from agent.py
from agent import app, GraphState
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("✅ Agent imported successfully")
print(f"Azure Endpoint configured: {os.getenv('PIVOTLY_AZURE_OPENAI_ENDPOINT') is not None}")
print(f"Azure API Key configured: {os.getenv('PIVOTLY_AZURE_OPENAI_API_KEY') is not None}")

✅ Agent imported successfully
Azure Endpoint configured: True
Azure API Key configured: True


## Test 1: Simple Sum Calculation

Test the agent with a basic question about summing numbers.

In [3]:
# Test case 1: Sum of numbers
test_state = {
    "messages": [("human", "Write code that takes a list of numbers and returns their sum")],
    "error": "",
    "generation": "",
    "iterations": 0,
    "execution_method": "exec",
    "test_data": {'numbers': [1, 2, 3]}
}

result = app.invoke(test_state)

print("📊 Results:")
print(f"  Generated code: {result.get('generation', 'No generation')}")
print(f"  Error: {result.get('error', 'None')}")
print(f"  Iterations: {result.get('iterations', 0)}")

LLM created successfully
Attempt 1
Executing code: sum(numbers)
Test data: {'numbers': [1, 2, 3]}
Attempt 1 succeeded with result: 6
📊 Results:
  Generated code: sum(numbers)
  Error: no
  Iterations: 1


## Test 2: Sum of Squares

Test with a more complex calculation.

In [6]:
# Test case 2: Sum of squares
test_state_2 = {
    "messages": [("human", "Write code that calculates the sum of squares of numbers in a list")],
    "error": "",
    "generation": "",
    "iterations": 0,
    "execution_method": "exec",
    "test_data": {'numbers': [1, 2, 3]}
}

result_2 = app.invoke(test_state_2)

print("📊 Results:")
print(f"  Generated code: {result_2.get('generation', 'No generation')}")
print(f"  Error: {result_2.get('error', 'None')}")
print(f"  Iterations: {result_2.get('iterations', 0)}")

LLM created successfully
Attempt 1
Executing code: sum(x*x for x in numbers)
Test data: {'numbers': [1, 2, 3]}
Attempt 1 succeeded with result: 14
📊 Results:
  Generated code: sum(x*x for x in numbers)
  Error: no
  Iterations: 1


## Test 3: Using Subprocess Execution Method

Test with subprocess execution method for better isolation.

In [5]:
# Test case 3: Using subprocess method
test_state_3 = {
    "messages": [("human", "Write code that finds the maximum number in a list")],
    "error": "",
    "generation": "",
    "iterations": 0,
    "execution_method": "subprocess",  # Use subprocess for isolation
    "test_data": {'numbers': [1, 2, 3]}
}

result_3 = app.invoke(test_state_3)

print("📊 Results:")
print(f"  Generated code: {result_3.get('generation', 'No generation')}")
print(f"  Error: {result_3.get('error', 'None')}")
print(f"  Iterations: {result_3.get('iterations', 0)}")

LLM created successfully
Attempt 1
Attempt 1 succeeded with result: 3
📊 Results:
  Generated code: max(numbers) if numbers else None
  Error: no
  Iterations: 1


## Interactive Testing Function

Create a helper function to easily test different questions.

In [4]:
def test_agent(question: str, execution_method: str = "exec", test_data: dict = None):
    """
    Test the agent with a custom question.
    
    Args:
        question: The natural language question to ask
        execution_method: Either "exec" or "subprocess"
        test_data: Optional test data dictionary. If None, uses default {'numbers': [1, 2, 3]}
    """
    # Build test_state, only include test_data if it's provided
    test_state = {
        "messages": [("human", question)],
        "error": "",
        "generation": "",
        "iterations": 0,
        "execution_method": execution_method
    }
    
    # Only add test_data if it's provided (not None)
    if test_data is not None:
        test_state["test_data"] = test_data
    
    print(f"🤔 Question: {question}")
    print(f"⚙️  Execution method: {execution_method}")
    if test_data:
        print(f"📋 Test data: {test_data}")
    else:
        print("📋 Test data: Using default {'numbers': [1, 2, 3]}")
    print("🔄 Processing...\n")
    
    result = app.invoke(test_state)
    
    print("📊 Results:")
    print(f"  ✅ Generated code: {result.get('generation', 'No generation')}")
    error_msg = result.get('error', 'None')
    error_icon = '❌' if error_msg and error_msg != 'no' else '✅'
    print(f"  {error_icon} Error: {error_msg}")
    print(f"  🔁 Iterations: {result.get('iterations', 0)}")
    
    # Show execution result if available in the result
    if 'result' in result:
        print(f"  📈 Execution result: {result.get('result', 'N/A')}")
    
    return result


In [5]:
test_data = {
    "x_values": [2, 4, 6, 8, 10],
    "y_values": [1, 3, 5, 7, 9]
}

result = test_agent(
    "Write code that calculates x^2 + y^2 for each x and y in the list",
    execution_method="subprocess",
    test_data=test_data
)

🤔 Question: Write code that calculates x^2 + y^2 for each x and y in the list
⚙️  Execution method: subprocess
📋 Test data: {'x_values': [2, 4, 6, 8, 10], 'y_values': [1, 3, 5, 7, 9]}
🔄 Processing...

LLM created successfully
Attempt 1
Attempt 1 succeeded with result: [5, 25, 61, 113, 181]
📊 Results:
  ✅ Generated code: [x**2 + y**2 for x, y in zip(x_values, y_values)]
  ✅ Error: no
  🔁 Iterations: 1
